Preprocessing notebook for Bristol Airbnb listings data
*Co-authored with CoCo*

# Import Libraries

In [ ]:
import pandas as pd
import ast
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import call_function, col

session = get_active_session()

bristol_listings_sp = session.table("practice_airbnb.public.bristol_listings")

# Listings Table

In [ ]:
def listings_geography(df_sp):
    df = df_sp.with_column(
        "geography",
        call_function("st_makepoint", col('"longitude"'), col('"latitude"'))
    )
    df = df.to_pandas()
    return df

In [ ]:
#Hosts table created from all the host columns from listings
def make_hosts(df):
    hosts = df[['host_id',
                  'host_url',
                  'host_name',
                  'host_since',
                  'host_location',
                  'host_about',
                  'host_response_time',
                  'host_response_rate',
                  'host_acceptance_rate',
                  'host_is_superhost',
                  'host_thumbnail_url',
                  'host_picture_url',
                  'host_neighbourhood',
                  'host_listings_count',
                  'host_total_listings_count',
                  'host_verifications',
                  'host_has_profile_pic',
                  'host_identity_verified']]
    return hosts

In [ ]:
#All columns deemed unnecessary are dropped
def drop_listings_columns(df):
    df = df.drop(axis=1,columns=['neighbourhood',
                                          'latitude',
                                          'longitude',
                                          'bathrooms_text',
                                          'beds',
                                          'availability_30',
                                          'availability_60',
                                          'availability_90',
                                          'number_of_reviews_ltm',
                                          'number_of_reviews_l30d',
                                          'availability_eoy',
                                          'number_of_reviews_ly',
                                          'host_url',
                                          'host_name',
                                          'host_since',
                                          'host_location',
                                          'host_about',
                                          'host_response_time',
                                          'host_response_rate',
                                          'host_acceptance_rate',
                                          'host_is_superhost',
                                          'host_thumbnail_url',
                                          'host_picture_url',
                                          'host_neighbourhood',
                                          'host_listings_count',
                                          'host_total_listings_count',
                                          'host_verifications',
                                          'host_has_profile_pic',
                                          'host_identity_verified',
                                          'scrape_id',
                                          'last_scraped',
                                          'source',
                                          'neighbourhood_group_cleansed',
                                          'minimum_nights',
                                          'maximum_nights',
                                          'minimum_minimum_nights',
                                          'maximum_minimum_nights',
                                          'minimum_maximum_nights',
                                          'maximum_maximum_nights',
                                          'minimum_nights_avg_ntm',
                                          'maximum_nights_avg_ntm',
                                          'calendar_last_scraped',
                                          'license',
                                          'instant_bookable',
                                          'calculated_host_listings_count',
                                          'calculated_host_listings_count_entire_homes',
                                          'calculated_host_listings_count_private_rooms',
                                          'calculated_host_listings_count_shared_rooms',
                                          'has_availability',
                                          'calendar_updated',
                                          'neighborhood_overview',
                                          'first_review',
                                          'last_review',
                                          'reviews_per_month'])
    return df

In [ ]:
def listings_clean(df):
    #All rows with any nulls are not useful
    df = df.dropna(axis='rows',how='any')
    #Remove $ and , characters so that price can be converted to float type
    df['price'] = df['price'].str.replace('[$,]', '', regex=True)
    #Reduce duplication between room_type and property_type
    df['property_type'] = df['property_type'].str.replace(r'Entire |Private room in ','',regex=True).str.title()
    return df

In [ ]:
def explode_amenities(df):
    df['amenities'] = df['amenities'].apply(ast.literal_eval)
    
    exploded = df['amenities'].explode()
    dummies = pd.crosstab(exploded.index, exploded).clip(upper=1)
    return (dummies,exploded)

In [ ]:
def simplify_amenity(a):
    if not isinstance(a, str):
        return a
    a_lower = a.lower()
    if 'hdtv' in a_lower or a_lower.endswith('tv') or ' tv' in a_lower:
        return 'TV'
    if 'wifi' in a_lower:
        return 'Wifi'
    if 'parking' in a_lower:
        return 'Parking'
    if 'sound system' in a_lower:
        return 'Sound System'
    if 'oven' in a_lower:
        return 'Oven'
    if 'stove' in a_lower:
        return 'Stove'
    if 'bbq' in a_lower or 'barbeque' in a_lower:
        return 'Barbeque'
    if 'refrigerator' in a_lower:
        return 'Refrigerator'
    if 'conditioner' in a_lower or 'shampoo' in a_lower or 'soap' in a_lower or 'shower gel' in a_lower:
        return 'Toiletries'
    if 'baby' in a_lower:
        return 'Baby Friendly'
    if 'backyard' in a_lower:
        return 'Backyard'
    if 'booster seat' in a_lower or 'high chair':
        return 'Booster Seat/High Chair'
    if 'changing table' in a_lower:
        return 'Changing Table'
    if 'children\'s books and toys' in a_lower:
        return 'Childrens Books and Toys'
    if 'clothing storage' in a_lower:
        return 'Clothing Storage'
    if 'coffee maker' in a_lower:
        return 'Coffee Maker'
    if 'ev charger' in a_lower:
        return 'EV Charger'
    if 'exercise equipment' in a_lower or 'gym' in a_lower:
        return 'Exercise Equipment'
    if 'dryer' in a_lower:
        return 'Dryer'
    if 'dishwasher' in a_lower or 'washer' in a_lower:
        return 'Dishwasher'
    if 'game console' in a_lower:
        return 'Game Console'
    if 'housekeeping' in a_lower:
        return 'Housekeeping'
    if 'indoor fireplace' in a_lower:
        return 'Indoor Fireplace'
    if 'hot tub' in a_lower:
        return 'Hot Tub'
    if 'pool' in a_lower:
        return 'Pool'
    if 'conditioning' in a_lower or 'ac' in a_lower:
        return 'Air Conditioning'
    return a

# Host Table

In [ ]:
def hosts_clean(df):
    df = df.drop_duplicates()
    #WIP MORE TO BE ADDED
    return df

# Full Preprocessing

In [ ]:
def preprocess_listings(df):
    df = listings_geography(df)
    hosts = make_hosts(df)

    df = drop_listings_columns(df)
    df = listings_clean(df)
    hosts = hosts_clean(hosts)

    #Exploding amenities column into multiple columns
    dummies,exploded = explode_amenities(df)
    exploded_clean = exploded.apply(simplify_amenity)
    dummies = pd.crosstab(exploded_clean.index, exploded_clean).clip(upper=1)
    df = df.join(dummies)

    return (df,hosts)

bristol_listings_clean,bristol_hosts_clean = preprocess_listings(bristol_listings_sp)

# Outliers

In [ ]:
def null_percent(df, column):
    pct = df[column].isnull().sum() / len(df)*100
    return pct

In [ ]:
def outlier_count(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    return ((df[column] < Q1 - 1.5 * IQR) | (df[column] > Q3 + 1.5 * IQR)).sum()

In [ ]:
def append_dd(df):
    rows = []
    for column in df.columns:
        is_numeric = pd.api.types.is_numeric_dtype(df[column])
        rows.append({
            'Column Name': column,
            'Data Type': str(df[column].dtype),
            'Percent Null': null_percent(df, column),
            'Range': f"{df[column].max() - df[column].min()}" if is_numeric else None,
            'Number of Unique Values': df[column].nunique(),
            'Mean': f"{df[column].mean()}" if is_numeric else None,
            'Median': f"{df[column].median()}" if is_numeric else None,
            'Standard Deviation': f"{df[column].std()}" if is_numeric else None,
            'Max': f"{df[column].max()}" if is_numeric else None,
            'Min': f"{df[column].min()}" if is_numeric else None,
            'Skew': f"{df[column].skew()}" if is_numeric else None,
            'Outlier Count': f"{outlier_count(df, column)}" if is_numeric else None
        })
    return pd.DataFrame(rows)

In [ ]:
# Convert boolean columns to int so numeric stats work correctly
#bool_cols = bristol_hosts.select_dtypes(include='bool').columns
#bristol_hosts[bool_cols] = bristol_hosts[bool_cols].astype(int)
#bh = append_dd(bristol_hosts)
#bh

In [ ]:
#bl = append_dd(bristol_listings)
#bl[['Column Name','Outlier Count','Range','Mean','Median','Min','Max']]

In [ ]:
bristol_listings_clean.info()